# API Data Ingest Practice

**Level:** Beginner  
**Tags:** API, Data, Workflow

## Overview

Pull free market data APIs, handle rate limits, normalize columns

## References

1. Hull, J.C. (2022). *Options, Futures, and Other Derivatives*. Pearson.
2. Shreve, S.E. (2004). *Stochastic Calculus for Finance II*. Springer.
3. Wilmott, P. (2006). *Paul Wilmott on Quantitative Finance*. Wiley.

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

# Additional imports
try:
    import yfinance as yf
    print('yfinance imported successfully')
except ImportError:
    print('yfinance not installed. Install with: pip install yfinance')
    yf = None

import requests
from datetime import datetime, timedelta
import time

# Configuration
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline
np.random.seed(42)

print('Libraries imported successfully!')

## 1. Introduction to API Data Ingest Practice

In quantitative finance, accessing real-time and historical market data is essential for analysis, modeling, and backtesting. Many free and paid APIs provide financial data such as stock prices, market indicators, and economic data.

This notebook introduces:
- Connecting to popular financial APIs
- Handling rate limits and errors
- Data normalization and cleaning
- Basic visualization of ingested data

We'll use free APIs from Yahoo Finance (via yfinance library) and Alpha Vantage as examples.

### Installing Required Libraries

If not already installed, run the following commands:

```bash
!pip install yfinance requests pandas
```

- **yfinance**: Wrapper for Yahoo Finance API
- **requests**: For making HTTP requests to APIs
- **pandas**: For data manipulation (already imported)

In [ ]:
# Function to fetch data using yfinance
def fetch_yahoo_data(tickers, start_date='2020-01-01', end_date=None):
    """
    Fetch historical data for given tickers using Yahoo Finance.
    
    Parameters:
    - tickers: list of ticker symbols (e.g., ['AAPL', 'GOOG'])
    - start_date: start date in YYYY-MM-DD format
    - end_date: end date in YYYY-MM-DD format (default: today)
    
    Returns:
    - pd.DataFrame with multi-index columns
    """
    if yf is None:
        print('yfinance not available')
        return None
    
    if end_date is None:
        end_date = datetime.today().strftime('%Y-%m-%d')
    
    try:
        data = yf.download(tickers, start=start_date, end=end_date)
        print(f'Successfully fetched data for {len(tickers)} tickers')
        return data
    except Exception as e:
        print(f'Error fetching data: {e}')
        return None

# Test the function
if yf is not None:
    sample_data = fetch_yahoo_data(['AAPL', 'MSFT'], '2023-01-01')
    print(sample_data.head() if sample_data is not None else 'No data fetched')
else:
    print('Skipping data fetch due to missing yfinance')

# Alternative: Using requests for Alpha Vantage (free tier has limits)
ALPHA_VANTAGE_API_KEY = 'YOUR_API_KEY_HERE'  # Get from https://www.alphavantage.co/support/#api-key

def fetch_alpha_vantage(symbol, function='TIME_SERIES_DAILY'):
    base_url = 'https://www.alphavantage.co/query'
    params = {
        'function': function,
        'symbol': symbol,
        'apikey': ALPHA_VANTAGE_API_KEY,
        'outputsize': 'compact'
    }
    
    try:
        response = requests.get(base_url, params=params)
        data = response.json()
        
        if 'Error Message' in data:
            print(f'API Error: {data["Error Message"]}')
            return None
        
        # Convert to DataFrame
        if function == 'TIME_SERIES_DAILY':
            df = pd.DataFrame(data['Time Series (Daily)']).T
            df = df.astype(float)
            df.index = pd.to_datetime(df.index)
            df = df.sort_index()
            return df
        
        return data
    
    except Exception as e:
        print(f'Error fetching from Alpha Vantage: {e}')
        return None

print('Data fetching functions defined')

## 2. Mathematical Foundation

Data normalization is important to bring different scales to a common range, which is useful for analysis and visualization. Common methods include:

### Min-Max Normalization
Scales data to [0,1] range:

$$\hat{x} = \frac{x - \min(x)}{\max(x) - \min(x)}$$

### Standardization (Z-score)
Centers data around mean with unit variance:

$$\hat{x} = \frac{x - \mu}{\sigma}$$

where $\mu$ is the mean and $\sigma$ is the standard deviation.

### Percentage Change (Returns)
For financial time series:

$$r_t = \frac{P_t - P_{t-1}}{P_{t-1}} \times 100$$

### Exponential Moving Average
For smoothing:

$$EMA_t = \alpha \times P_t + (1 - \alpha) \times EMA_{t-1}$$

These formulas help preprocess and analyze financial data from APIs.

## 3. Python Implementation

Now let's implement data normalization and cleaning functions that we covered in the mathematical foundation.

In [ ]:
# Data normalization functions
def min_max_normalize(series):
    """Min-Max normalization to [0,1] range"""
    return (series - series.min()) / (series.max() - series.min())

def standardize(series):
    """Standardization (Z-score)"""
    return (series - series.mean()) / series.std()

def percentage_returns(prices):
    """Calculate percentage returns"""
    return prices.pct_change() * 100

def exponential_moving_average(series, alpha=0.1):
    """Calculate exponential moving average"""
    return series.ewm(alpha=alpha).mean()

# Data cleaning functions
def clean_price_data(data):
    """Clean OHLCV data by removing NaN values and forward filling"""
    # Forward fill missing values
    cleaned = data.fillna(method='ffill')
    # Remove any remaining NaN
    cleaned = cleaned.dropna()
    return cleaned

def calculate_technical_indicators(df):
    """Add simple technical indicators"""
    if 'Close' in df.columns:
        df = df.copy()
        df['Returns'] = percentage_returns(df['Close'])
        df['Normalized_Close'] = min_max_normalize(df['Close'])
        df['EMA_10'] = exponential_moving_average(df['Close'], 0.1)
    return df

print('Data processing functions defined')

## 4. Visualization and Analysis

Let's visualize the processed data and demonstrate the cleaning and normalization effects.

In [ ]:
# Demonstration with sample/fetched data
if 'sample_data' in globals() and sample_data is not None:
    # Assume single ticker for simplicity
    try:
        aapl_data = sample_data['AAPL'] if 'AAPL' in sample_data.columns.get_level_values(1) else None
        if aapl_data is None and len(sample_data.columns.get_level_values(1)) == 1:
            aapl_data = sample_data.iloc[:, 0]  # Single column
        
        if aapl_data is not None:
            # Clean and add indicators
            clean_aapl = clean_price_data(aapl_data)
            enriched_aapl = calculate_technical_indicators(clean_aapl)
            
            # Visualization
            fig, axes = plt.subplots(3, 2, figsize=(16, 12))
            
            # Price series
            clean_aapl['Close'].plot(ax=axes[0,0], title='Close Price')
            axes[0,0].set_ylabel('Price ($)')
            
            # Normalized price
            axes[0,1].plot(enriched_aapl.index, enriched_aapl['Normalized_Close'])
            axes[0,1].set_title('Normalized Close Price')
            axes[0,1].set_ylabel('Normalized Value')
            
            # Returns
            axes[1,0].plot(enriched_aapl.index, enriched_aapl['Returns'])
            axes[1,0].set_title('Daily Returns (%)')
            axes[1,0].set_ylabel('Returns (%)')
            
            # EMA
            axes[1,1].plot(clean_aapl.index, clean_aapl['Close'], label='Close')
            axes[1,1].plot(enriched_aapl.index, enriched_aapl['EMA_10'], label='EMA(10%)', color='red')
            axes[1,1].set_title('Price with EMA')
            axes[1,1].legend()
            axes[1,1].set_ylabel('Price ($)')
            
            # Volume
            if 'Volume' in clean_aapl.columns:
                axes[2,0].bar(clean_aapl.index, clean_aapl['Volume']/1e6)
                axes[2,0].set_title('Volume (Millions)')
                axes[2,0].set_ylabel('Volume (M)')
            
            # Returns distribution
            if not enriched_aapl['Returns'].empty:
                sns.histplot(enriched_aapl['Returns'].dropna(), ax=axes[2,1], kde=True)
                axes[2,1].set_title('Returns Distribution')
                axes[2,1].set_xlabel('Returns (%)')
            
            plt.tight_layout()
            plt.show()
            
            # Summary statistics
            print('\nSummary Statistics:')
            print(enriched_aapl[['Close', 'Returns', 'Normalized_Close', 'EMA_10']].describe())
            
        else:
            print('Could not extract single ticker data')
    except Exception as e:
        print(f'Error in visualization: {e}')
else:
    # Generate synthetic data for demonstration
    np.random.seed(42)
    dates = pd.date_range('2023-01-01', periods=252, freq='D')
    synthetic_prices = 150 + np.cumsum(np.random.normal(0.001, 0.02, 252)) * 150
    synthetic_data = pd.DataFrame({
        'Open': synthetic_prices * (1 + np.random.normal(0, 0.005, 252)),
        'High': synthetic_prices * (1 + np.random.normal(0, 0.01, 252)),
        'Low': synthetic_prices * (1 + np.random.normal(0, -0.01, 252)),
        'Close': synthetic_prices,
        'Volume': np.random.randint(1e6, 1e7, 252)
    }, index=dates)
    
    # Clean and enhance
    clean_synthetic = clean_price_data(synthetic_data)
    enriched_synthetic = calculate_technical_indicators(clean_synthetic)
    
    # Basic visualization
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # Close price
    axes[0,0].plot(clean_synthetic.index, clean_synthetic['Close'])
    axes[0,0].set_title('Synthetic Close Price')
    axes[0,0].set_ylabel('Price ($)')
    
    # Normalized price
    axes[0,1].plot(enriched_synthetic.index, enriched_synthetic['Normalized_Close'])
    axes[0,1].set_title('Normalized Close Price')
    
    # Returns
    axes[1,0].plot(enriched_synthetic.index, enriched_synthetic['Returns'])
    axes[1,0].set_title('Daily Returns (%)')
    
    # Returns distribution
    sns.histplot(enriched_synthetic['Returns'].dropna(), ax=axes[1,1], kde=True)
    axes[1,1].set_title('Returns Distribution')
    axes[1,1].set_xlabel('Returns (%)')
    
    plt.tight_layout()
    plt.show()
    
    print('\nSynthetic data used for demonstration. Replace with real API data.')

## 5. Practical Applications

This data ingestion and processing approach is fundamental for:

- **Backtesting Trading Strategies**: Need clean, normalized historical data
- **Portfolio Analysis**: Compare assets on common scales
- **Risk Management**: Calculate returns distributions and volatility measures
- **Machine Learning Models**: Preprocessed features for prediction models
- **Real-time Trading Systems**: Similar processing for live data feeds

**Best Practices:**
- Always handle missing data appropriately (forward/backward fill vs drop)
- Respect API rate limits by implementing delays between requests
- Cache data locally to avoid repeated API calls
- Validate data quality and handle API errors gracefully
- Use appropriate normalization based on your analysis needs (e.g., standardization for ML)

## Summary

This notebook covered API data ingest practice:

- Implemented functions to fetch data from Yahoo Finance and Alpha Vantage APIs
- Added data normalization (min-max, standardization) and cleaning utilities
- Demonstrated visualization of price data, returns, and technical indicators
- Created technical indicators like EMA and returns calculations

Key takeaways:
- Use try-except blocks for robust API calls
- Data cleaning is crucial before analysis
- Normalization helps compare different scale data
- Visualization aids understanding of financial time series

### Next Steps
- Experiment with real API keys for live data
- Implement additional technical indicators
- Build a simple trading backtest using ingested data
- Add more APIs (Quandl, IEX Cloud, etc.)

### Additional Resources

- [yfinance Documentation](https://pypi.org/project/yfinance/)
- [Alpha Vantage API](https://www.alphavantage.co/documentation/)
- [pandas Documentation](https://pandas.pydata.org/docs/)
- [QuantEdX Courses](https://www.quantedx.com/courses)
- [Community Forum](https://www.quantedx.com/community)
- [GitHub Repository](https://github.com/quantedx)